# FPGA implementation

$\renewcommand{\ket}[1]{\left|#1\right\rangle}\renewcommand{\bra}[1]{\left\langle #1\right|}\renewcommand{\braket}[2]{\left\langle #1 \middle| #2 \right\rangle}\renewcommand{\ketbra}[2]{\left|#1\right\rangle\!\left\langle #2\right|}$

This notebook explains how the FPGA timing estimates for the classical MCMC proposal kernels are obtained.

Among the classical backends considered here, the FPGA implementation is the closest analogue of the quantum-circuit implementation. For each system size, the computation is synthesized as a fixed hardware datapath rather than executed as a sequence of instructions through a CPU or GPU memory hierarchy. The amount of hardware therefore grows with the system size, while the critical-path latency grows much more slowly.

The kernels are implemented in hardware-oriented C++ using AMD Vitis High-Level Synthesis (HLS). Arrays, loops, and helper functions are translated into hardware structures according to the HLS directives, and the resulting design is synthesized into an RTL representation.

**Table of contents**

1. [Source](#source)
2. [Algorithm](#algorithm)
   1. [Local move](#local-move)
   2. [Uniform move](#uniform-move)
3. [Timing calculation](#timing-calculation)
4. [Reproducibility](#reproducibility)

<a id="source"></a>
## Source

The code and data are available in the `estimation_timing/fpga` folder.

The folder contains:

* `src/`: HLS source files. `local_move_hls.cpp` and `uniform_move_hls.cpp` implement the circuits used to estimate the acceptance probabilities for the local and uniform proposal moves. The generated header `exp_cheby.hpp` contains the Chebyshev coefficients used to approximate the exponential acceptance function and is produced by `make_exp_cheby.py`.
* `tb/`: C-simulation testbenches. `tb_local_move_n10_all.cpp` checks the local circuit for all $2^{10}$ configurations and all ten single-spin flips. `tb_uniform_move_n10_all.cpp` checks the uniform circuit for all $2^{10}$ old configurations and a deterministic set of ten proposed configurations per old configuration.
* `scripts/`: Tcl scripts used by the AMD toolchain. `run_synthesis_move.tcl` runs synthesis, while `run_csim_local_n10.tcl` and `run_csim_uniform_n10.tcl` run the testbenches.
* `logs/`: logs generated by the execution scripts.
* `reports/`: synthesis reports for the local and uniform circuits at $n\in\{8,16,32,64\}$. These reports provide the latency and resource estimates used below.
* `launch_testbench_local_n10.sh`: runs the local-move testbench.
* `launch_testbench_uniform_n10.sh`: runs the uniform-move testbench.
* `launch_synthesis_many_n.sh`: synthesizes both proposal circuits for $n\in\{8,16,32,64\}$.
* `copy_synthesis_reports.sh`: copies the synthesis reports from the generated implementation directories into `reports/`.
* `launch_visualization.sh`: generates a reduced $n=8$ implementation with eight fractional bits for visualization.
* `visualization/`: reduced synthesized implementations, generated Verilog, Yosys outputs, and manually prepared TikZ architecture diagrams.

<a id="algorithm"></a>
## Algorithm


<a id="local-move"></a>
### FPGA local move implementation

The local FPGA kernel implements the Metropolis acceptance logic for a single-spin-flip proposal. The current configuration is fixed, and a one-hot mask selects the spin $i$ to flip. Since only one spin changes, the circuit does not recompute the complete SK energy. Instead, it uses

$$
\frac{\Delta_i H}{\Lambda}
=
2x_i
\left(
\frac{\widetilde h_i}{\Lambda}
+
\sum_{j\neq i}
\frac{\widetilde J_{ij}}{\Lambda}x_j
\right),
$$

where $\Lambda$ is the upper bound on the energy difference used in the fixed-point model.

The coefficients $\widetilde h_i$ and $\widetilde J_{ij}$ enter the circuit after normalization by $\Lambda$. This allows the normalized coefficients, partial sums, and normalized energy difference to be represented using one sign bit and fractional bits. The terms $(\widetilde J_{ij}/\Lambda)x_j$ are generated in parallel and reduced through a logarithmic-depth adder tree. The latency therefore grows as $O(\log n)$, while the hardware area grows as $O(n)$.

The circuit then computes the positive part

$$
x
=
\frac{(\Delta_i H)_+}{\Lambda}
=
\max\left\{
\frac{\Delta_i H}{\Lambda},
0
\right\}.
$$

Downhill moves have $x=0$ and are accepted with probability one. Uphill moves use the factor $\exp(-\beta\Lambda x)$. This exponential is approximated by a truncated Chebyshev series evaluated using the Clenshaw recurrence. The polynomial is evaluated only up to the cutoff

$$
X_{\mathrm{cut}}
=
\min\left\{
1,
\frac{\log(\varepsilon_{\mathrm{tail}}^{-1})}{\beta\Lambda}
\right\},
$$

and the approximation returns zero above this cutoff.

The selected clock period is primarily limited by the fixed-point multiplications in the polynomial block. These multiplications could be pipelined or split into multicycle operations to target a higher clock frequency, at the cost of additional latency cycles.

**Datatypes and constants**

* `SK_N`: compile-time number of spins, denoted by $n$. The visualization uses `SK_N = 8`.
* `FRAC`: compile-time number of fractional bits, denoted by $b$. The visualization uses `FRAC = 8`.
* `NUM_J`: number of independent couplings $\widetilde J_{ij}$ with $i<j$, equal to $n(n-1)/2$. For $n=8$, `NUM_J = 28`.
* `coeff_t`: signed fixed-point type $\langle 1.b\rangle$ storing $\widetilde h_i/\Lambda$ and $\widetilde J_{ij}/\Lambda$.
* `delta_t`: signed fixed-point type $\langle 1.b\rangle$ storing normalized local fields, normalized energy differences, partial sums, and $x$.
* `prob_t`: unsigned fixed-point type $\langle 1.31\rangle$ storing the acceptance probability in $[0,1]$.
* `poly_t`: signed fixed-point type $\langle 4.36\rangle$ storing Chebyshev coefficients and intermediate values in the Clenshaw recurrence.
* `scale_t`: unsigned fixed-point type $\langle 16.32\rangle$ storing $X_{\mathrm{cut}}^{-1}$.
* `spins`: $n$-bit string representing the current spin configuration, with $0\mapsto-1$ and $1\mapsto+1$.
* `flip_mask`: one-hot $n$-bit string selecting the spin $i$ to flip.
* `EXP_X_CUT`: cutoff $X_{\mathrm{cut}}$ generated offline in `exp_cheby.hpp`.
* `EXP_INV_X_CUT`: inverse cutoff $X_{\mathrm{cut}}^{-1}$ generated offline in `exp_cheby.hpp`.
* `CHEB_COEFFS`: Chebyshev coefficients generated offline in `exp_cheby.hpp`.

**Inputs**

* `h`: vector of $n$ `coeff_t` values storing $\widetilde h_i/\Lambda$.
* `J_edges`: vector of `NUM_J` `coeff_t` values storing $\widetilde J_{ij}/\Lambda$ in packed upper-triangular form.
* `spins`: $n$-bit string representing the current configuration.
* `flip_mask`: one-hot $n$-bit string selecting the spin to flip.

**Outputs**

* `delta_x`: pointer to a `delta_t` value containing $x=(\Delta_iH)_+/\Lambda$.
* `accept_prob`: pointer to a `prob_t` value containing the approximation to $\exp(-\beta\Lambda x)$.

**Methods**

* `spin_value`: converts a spin bit into an Ising value in $\{-1,+1\}$.
* `edge_index`: maps a pair $(i,j)$ to the packed upper-triangular index used to access $\widetilde J_{ij}$.
* `selected_field`: uses `flip_mask` as a one-hot multiplexer to select $\widetilde h_i$.
* `selected_coupling_for_j`: uses `flip_mask` to select $\widetilde J_{ij}$ for the chosen spin.
* `selected_spin`: extracts the selected spin $x_i$ from `spins` and `flip_mask`.
* `tree_sum<>`: implements a fully unrolled binary adder tree. The local circuit reduces one field term and $n$ coupling slots, including the zero self-coupling slot.
* `local_delta_over_alpha`: computes the normalized energy difference. The function name retains `alpha` for compatibility with the implementation, but the normalization scale corresponds to $\Lambda$ in the manuscript.
* `positive_part`: computes $x=(\Delta_iH)_+/\Lambda$.
* `chebyshev_g`: evaluates the piecewise Chebyshev approximation to $\exp(-\beta\Lambda x)$ using the Clenshaw recurrence.
* `local_spin_flip_operation`: top-level HLS function combining the local energy-difference calculation, clipping, and acceptance-probability approximation.

![Schematic representation of the datapath for the local-move circuit](../estimation_timing/fpga/visualization/schematics/local_N8_cpp_architecture_tikz.png)

The figure shows the local-move datapath for $n=8$ and $b=8$. Blue boxes denote top-level inputs and outputs, green boxes denote helper blocks, and red boxes denote the main arithmetic blocks. Solid arrows denote runtime data flow. The circuit selects the relevant $\widetilde h_i$, $\widetilde J_{ij}$, and $x_i$, forms the local-field terms, and reduces them through a logarithmic-depth adder tree. It then computes $\Delta_iH/\Lambda$, clips it to $x=(\Delta_iH)_+/\Lambda$, and evaluates the acceptance probability.


<a id="uniform-move"></a>
### FPGA uniform move implementation

The uniform FPGA kernel implements the Metropolis acceptance logic for a dense nonlocal proposal. The proposed configuration may differ from the current configuration at many spin positions, so the local single-spin-flip identity cannot be used. The circuit therefore computes the complete normalized SK energy of the proposed configuration,

$$
\frac{H(y)}{\Lambda}
=
-\sum_i\frac{\widetilde h_i}{\Lambda}y_i
-\sum_{i<j}\frac{\widetilde J_{ij}}{\Lambda}y_i y_j,
$$

and subtracts the supplied normalized energy $H(x)/\Lambda$ of the current configuration.

As in the local circuit, the coefficients enter after normalization by $\Lambda$. The field and interaction terms are generated in parallel and reduced through a logarithmic-depth adder tree. The reduction contains

$$
n+\binom{n}{2}
=
\frac{n(n+1)}{2}
$$

terms. Its depth is $O(\log n)$, while its hardware area scales as $O(n^2)$.

**Datatypes and constants**

* `SK_N`: compile-time number of spins, denoted by $n$. The visualization uses `SK_N = 8`.
* `FRAC`: compile-time number of fractional bits, denoted by $b$. The visualization uses `FRAC = 8`.
* `NUM_J`: number of independent couplings $\widetilde J_{ij}$ with $i<j$, equal to $n(n-1)/2$. For $n=8$, `NUM_J = 28`.
* `NUM_ENERGY_TERMS`: number of terms in the dense energy, equal to $n+\mathrm{NUM\_J}=n(n+1)/2$. For $n=8$, `NUM_ENERGY_TERMS = 36`.
* `coeff_t`: signed fixed-point type $\langle 1.b\rangle$ storing $\widetilde h_i/\Lambda$ and $\widetilde J_{ij}/\Lambda$.
* `delta_t`: signed fixed-point type $\langle 1.b\rangle$ storing normalized energies, normalized energy differences, intermediate partial sums, and the clipped value $x$.
* `prob_t`: unsigned fixed-point type $\langle 1.31\rangle$ storing the acceptance probability in $[0,1]$.
* `poly_t`: signed fixed-point type $\langle 4.36\rangle$ storing Chebyshev coefficients and Clenshaw intermediate values.
* `scale_t`: unsigned fixed-point type $\langle 16.32\rangle$ storing $X_{\mathrm{cut}}^{-1}$.
* `new_spins`: $n$-bit string representing the proposed configuration.
* `old_energy`: signed fixed-point value storing $H(x)/\Lambda$.
* `EXP_X_CUT`: cutoff $X_{\mathrm{cut}}$ generated offline in `exp_cheby.hpp`.
* `EXP_INV_X_CUT`: inverse cutoff $X_{\mathrm{cut}}^{-1}$ generated offline in `exp_cheby.hpp`.
* `CHEB_COEFFS`: Chebyshev coefficients generated offline in `exp_cheby.hpp`.

**Inputs**

* `h`: vector of $n$ `coeff_t` values storing $\widetilde h_i/\Lambda$.
* `J_edges`: vector of `NUM_J` `coeff_t` values storing $\widetilde J_{ij}/\Lambda$ in packed upper-triangular form.
* `new_spins`: $n$-bit string representing the proposed configuration $y$.
* `old_energy`: `delta_t` value containing $H(x)/\Lambda$.

**Outputs**

* `new_energy`: pointer to a `delta_t` value containing $H(y)/\Lambda$.
* `delta_x`: pointer to a `delta_t` value containing $x=(H(y)-H(x))_+/\Lambda$.
* `accept_prob`: pointer to a `prob_t` value containing the approximation to $\exp(-\beta\Lambda x)$.

**Methods**

* `spin_value`: converts each bit of `new_spins` into an Ising spin in $\{-1,+1\}$.
* `edge_index`: maps each pair $(i,j)$ to the packed upper-triangular index used to access $\widetilde J_{ij}$.
* `field_delta_term`: computes each field contribution $-\widetilde h_i y_i/\Lambda$.
* `pair_delta_term`: computes each interaction contribution $-\widetilde J_{ij}y_i y_j/\Lambda$.
* `tree_sum<>`: implements a fully unrolled binary adder tree over all dense-energy terms.
* `dense_energy_over_alpha`: computes $H(y)/\Lambda$. The function name retains `alpha` for compatibility with the implementation, but the normalization scale corresponds to $\Lambda$ in the manuscript.
* `positive_part`: computes $x=(H(y)-H(x))_+/\Lambda$.
* `chebyshev_g`: evaluates the piecewise Chebyshev approximation to $\exp(-\beta\Lambda x)$.
* `uniform_move_operation`: top-level HLS function computing the proposed energy, subtracting the current energy, clipping the difference, and evaluating the acceptance probability.

![Schematic representation of the datapath for the uniform-move circuit](../estimation_timing/fpga/visualization/schematics/uniform_N8_cpp_architecture_tikz.png)

The figure shows the uniform-move datapath for $n=8$ and $b=8$. The color convention matches the local circuit. The circuit computes all field terms and interaction terms in parallel, reduces them through a logarithmic-depth adder tree, subtracts the supplied current energy, clips the result to $x=(H(y)-H(x))_+/\Lambda$, and evaluates the Metropolis acceptance probability.


<a id="timing-calculation"></a>
## Timing calculation

The timing information is extracted from the Vitis HLS synthesis reports in the `reports/` folder. The relevant files are the `local_spin_flip_operation_csynth*` reports for the local move and the `uniform_move_operation_csynth*` reports for the uniform move. Vitis produces both human-readable and machine-readable reports.

The selected clock period is $3.00\,\mathrm{ns}$, corresponding to a target frequency of approximately $333.3\,\mathrm{MHz}$. The critical path is primarily limited by the wide fixed-point multiplications in the piecewise Chebyshev block. Pipelining or splitting these multiplications into multicycle operators could permit a higher target frequency, but would increase the number of latency cycles.

For the local proposal, the energy difference contains one selected field term and $n-1$ interaction terms, giving $O(n)$ terms. For the uniform proposal, the complete dense energy contains $n$ field terms and $n(n-1)/2$ interaction terms, giving $n(n+1)/2=O(n^2)$ terms.

The following tables report the synthesis results.

**Local move**

| $n$ | Fractional bits $b$ | Energy-difference terms | Target clock (ns) | Latency (cycles) | Latency ($\mu$s) | DSP | FF | LUT |
|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 8  | 24 | 8  | 3.00 | 91 | 0.273 | 50  | 6,963  | 11,132 |
| 16 | 28 | 16 | 3.00 | 92 | 0.276 | 146 | 11,719 | 21,318 |
| 32 | 31 | 32 | 3.00 | 92 | 0.276 | 274 | 20,974 | 54,540 |
| 64 | 35 | 64 | 3.00 | 93 | 0.279 | 530 | 47,471 | 190,598 |

**Uniform move**

| $n$ | Fractional bits $b$ | Dense-energy terms | Target clock (ns) | Latency (cycles) | Latency ($\mu$s) | DSP | FF | LUT |
|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 8  | 24 | 36   | 3.00 | 89 | 0.267 | 18 | 4,975   | 16,728 |
| 16 | 28 | 136  | 3.00 | 90 | 0.270 | 18 | 9,436   | 48,876 |
| 32 | 31 | 528  | 3.00 | 92 | 0.276 | 18 | 28,128  | 183,801 |
| 64 | 35 | 2080 | 3.00 | 93 | 0.279 | 18 | 111,827 | 778,597 |

The latency in cycles is nearly constant over the synthesized sizes because the design exposes substantial spatial parallelism. Increasing $n$ primarily increases the number of instantiated arithmetic units and therefore the hardware area, rather than increasing the sequential latency.

The resource scaling differs substantially between the proposal rules. The local move requires $O(n)$ arithmetic terms, whereas the uniform move requires $O(n^2)$ terms. The logarithmic latency of the uniform proposal therefore assumes that the required quadratic hardware resources can be instantiated and is valid only within the capacity of the target FPGA.

Both transition times are fitted as affine functions of $\log_2 n$:

$$
\tau_{\mathrm{FPGA}}^{\mathrm{loc}}(n)
=
a_{\mathrm{FPGA}}^{\mathrm{loc}}
+
b_{\mathrm{FPGA}}^{\mathrm{loc}}\log_2 n,
$$

and

$$
\tau_{\mathrm{FPGA}}^{\mathrm{unif}}(n)
=
a_{\mathrm{FPGA}}^{\mathrm{unif}}
+
b_{\mathrm{FPGA}}^{\mathrm{unif}}\log_2 n.
$$

The following code fits the measured cycle counts and converts them to latency using the $3\,\mathrm{ns}$ clock period.


In [1]:
import numpy as np

clock_period_s = 3e-9
n_values = np.array([8, 16, 32, 64], dtype=float)
local_cycles = np.array([91, 92, 92, 93], dtype=float)
uniform_cycles = np.array([89, 90, 92, 93], dtype=float)


def fit_logarithmic_latency(
    n: np.ndarray,
    cycles: np.ndarray,
) -> tuple[float, float, np.ndarray]:
    """Fit cycles(n) = A + B log2(n)."""
    design_matrix = np.column_stack(
        [np.ones_like(n), np.log2(n)]
    )
    a_cycles, b_cycles = np.linalg.lstsq(
        design_matrix,
        cycles,
        rcond=None,
    )[0]
    fitted_cycles = design_matrix @ np.array(
        [a_cycles, b_cycles]
    )
    return a_cycles, b_cycles, fitted_cycles


a_local_cycles, b_local_cycles, fitted_local_cycles = (
    fit_logarithmic_latency(n_values, local_cycles)
)
a_uniform_cycles, b_uniform_cycles, fitted_uniform_cycles = (
    fit_logarithmic_latency(n_values, uniform_cycles)
)

a_local_s = clock_period_s * a_local_cycles
b_local_s = clock_period_s * b_local_cycles
a_uniform_s = clock_period_s * a_uniform_cycles
b_uniform_s = clock_period_s * b_uniform_cycles

print(
    "local cycles:  "
    f"{a_local_cycles:.3f} + "
    f"{b_local_cycles:.3f} log2(n)"
)
print(
    "uniform cycles:"
    f" {a_uniform_cycles:.3f} + "
    f"{b_uniform_cycles:.3f} log2(n)"
)
print(
    "local latency: "
    f"{a_local_s:.3e} + "
    f"{b_local_s:.3e} log2(n) s"
)
print(
    "uniform latency: "
    f"{a_uniform_s:.3e} + "
    f"{b_uniform_s:.3e} log2(n) s"
)


local cycles:  89.300 + 0.600 log2(n)
uniform cycles: 84.700 + 1.400 log2(n)
local latency: 2.679e-07 + 1.800e-09 log2(n) s
uniform latency: 2.541e-07 + 4.200e-09 log2(n) s


The fitted single-step latency models used in the paper are

$$
\tau_{\mathrm{FPGA}}^{\mathrm{loc}}(n)
=
\left(
0.2679
+
0.0018\log_2 n
\right)\,\mu\mathrm{s},
$$

and

$$
\tau_{\mathrm{FPGA}}^{\mathrm{unif}}(n)
=
\left(
0.2541
+
0.0042\log_2 n
\right)\,\mu\mathrm{s}.
$$

<a id="reproducibility"></a>
## Reproducibility

The FPGA workflow uses AMD Vitis/Vivado 2025.2. The AMD unified installer can be prepared with:

```bash
chmod +x FPGAs_AdaptiveSoCs_Unified_*_Lin64.bin
./FPGAs_AdaptiveSoCs_Unified_SDI_2025.2_1114_2157_Lin64.bin \
    --noexec \
    --target <installer-extraction-directory>
cd <installer-extraction-directory>
./xsetup -b ConfigGen
```

Edit the generated configuration file:

```bash
~/.Xilinx/install_config.txt
```

The relevant entries are:

```text
Destination=<installation-root>/xlnx
Modules=Virtex UltraScale+ FPGAs:1
```

Run the batch installation with:

```bash
./xsetup \
    --agree XilinxEULA,3rdPartyEULA \
    --batch Install \
    --config ~/.Xilinx/install_config.txt
```

Before each run, source the Vitis environment with `nounset` temporarily disabled:

```bash
set +u
source <installation-root>/xlnx/2025.2/Vitis/settings64.sh
set -u
```

This avoids failures when the AMD setup scripts reference initially unset variables such as `PYTHONPATH`. Check the installation with:

```bash
which vitis-run
which vivado
vitis --version
vivado -version
```

The Chebyshev coefficients are generated using Python, NumPy, and SciPy. The Python executable is invoked below as `python3.14`; it may instead be named `python3` in another configuration, and other compatible Python versions may also work.

```bash
python3.14 -m pip install numpy scipy
```

Run the local and uniform C-simulation testbenches with:

```bash
./launch_testbench_local_n10.sh \
    > logs/log_testbench_local_n10.txt
./launch_testbench_uniform_n10.sh \
    > logs/log_testbench_uniform_n10.txt
```

The local testbench fixes `SK_N = 10`, scans all $2^{10}$ configurations, and tests all ten one-hot local spin flips for each configuration. It compares `delta_x` with an independent floating-point calculation of $(\Delta_iH)_+/\Lambda$ and compares `accept_prob` with the corresponding exponential.

The uniform testbench also fixes `SK_N = 10`. It scans all $2^{10}$ current configurations and a deterministic set of ten proposed configurations for each current configuration. It compares `new_energy`, `delta_x`, and `accept_prob` with independent dense-energy and Metropolis calculations.

Run synthesis with:

```bash
./launch_synthesis_many_n.sh \
    > logs/log_launch_synthesis_many_n.txt
```

The synthesis launcher scans $n\in\{8,16,32,64\}$ and synthesizes both HLS top-level functions: `local_spin_flip_operation` and `uniform_move_operation`. For each $n$, the script computes the required number of fractional bits $b$, regenerates `src/exp_cheby.hpp`, and calls `scripts/run_synthesis_move.tcl`. The synthesis targets the `xcvu19p-fsva3824-2-e` FPGA with a clock period of $3.000\,\mathrm{ns}$.

Collect the reports with:

```bash
./copy_synthesis_reports.sh
```

To generate an RTL visualization with Yosys, run:

```bash
./launch_visualization.sh \
    > logs/log_launch_visualization.txt
```

This creates a reduced build with `SK_N = 8` and `FRAC = 8`. The script copies the HLS sources into `visualization/temp`, disables selected `#pragma HLS INLINE` directives, and extracts hierarchy-preserving Verilog into:

```text
visualization/hls_local_spin_flip_operation_N8_FRAC8_verilog/
visualization/hls_uniform_move_operation_N8_FRAC8_verilog/
```

Disabling inlining makes the generated RTL less optimized but easier to inspect because more function-level module boundaries are preserved. To generate schematic images, install Yosys, Graphviz, and `rsvg-convert`:

```bash
sudo apt-get install yosys graphviz librsvg2-bin
```

The generated schematics are stored in `visualization/schematics/`. The same folder also contains manually prepared LaTeX/TikZ diagrams that provide a cleaner architectural representation.
